# Phase 1 — Data analysis and feature engineering

Churn prevention agent for credit card customers.

**Churn here means the customer closed their credit card.** They remain a customer of the bank — they ended this one product. So the monthly purchase and payment columns are card activity.

---

This notebook is built **one step at a time.** Every step has three parts: a cell saying what we are about to do and why, the code, and a "What we gained" cell saying what it actually bought us.

The workflow is in `docs/Project_Workflow.pdf`. Every experiment with its numbers is in `docs/EXPERIMENT_LOG.md`.

## Step 0 — Load the data

Run this cell, then choose `bank_churn_dataset.csv` when the button appears.

In [ ]:
# --- Step 0: get the data file ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()          # choose bank_churn_dataset.csv
    DATA = list(uploaded.keys())[0]
else:
    DATA = "../data/raw/bank_churn_dataset.csv"

df = pd.read_csv(DATA)
print(df.shape[0], "customers,", df.shape[1], "columns")

## Step 1 — What is inside each column?

Before analysing anything, look at what the file actually holds. Three things for every column:

- **type** — the kind of value stored. `int64` is a whole number, `float64` is a number with decimals, `object` is text.
- **different values** — how many distinct values appear. Two means a yes/no column. Thousands means a measurement.
- **first customer** — one real example, so the numbers stop being abstract.

In [ ]:
# --- Step 1: what is inside each column? ---
overview = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "different values": df.nunique(),
    "first customer": df.iloc[0],
})

overview

### What we gained from Step 1

29 columns. Six describe the person, four the banking relationship, five the loans, twelve the six months of card activity, and one is the answer.

The twelve monthly columns are the only ones that describe *change*. Everything else is a single fact frozen at one moment. That shapes the whole of the rest of this phase.

## Step 2 — Is the data sound?

Three questions before trusting anything: are there empty cells, are any customers listed twice, and how many actually closed their card?

The third matters most. If only about 20% closed, then a model that predicts *"nobody ever leaves"* is correct 80% of the time and useless. That is why accuracy is never how this model gets judged.

Then two deeper checks. **Ranges** — is any value impossible on its own, like a negative age. And **contradictions** — is any value impossible *beside another*, like a customer who never had a loan but owes money on one.

In [ ]:
# --- Step 2a: basic quality check ---
print("rows, columns:", df.shape)
print("empty cells:", df.isnull().sum().sum())
print("duplicate rows:", df.duplicated().sum())
print()

counts = df["churned"].value_counts()
print("stayed:", counts[0])
print("closed:", counts[1])
print("closure rate:", round(100 * df["churned"].mean(), 1), "%")

In [ ]:
# --- Step 2b: range check — smallest and largest value per column ---
df.describe().T[["min", "max"]]

In [ ]:
# --- Step 2c: contradictions between columns ---
checks = {
    "never had a loan, but has a loan count":
        (df["had_loan_ever"] == 0) & (df["number_of_loans"] > 0),

    "never had a loan, but owes a balance":
        (df["had_loan_ever"] == 0) & (df["outstanding_loan_balance"] > 0),

    "never had a loan, but missed a loan payment":
        (df["had_loan_ever"] == 0) & (df["missed_loan_payment_ever"] == 1),

    "zero loans, but owes a balance":
        (df["number_of_loans"] == 0) & (df["outstanding_loan_balance"] > 0),

    "with the bank longer than they have been an adult":
        (df["loyalty_years"] > df["age"] - 18),
}

for name, rule in checks.items():
    print(f"{int(rule.sum()):>5}  {name}")

### What we gained from Step 2

Zero empty cells, zero duplicate rows, 6,803 stayed and 1,697 closed — a closure rate of 19.96%.

Every range is plausible and **all five contradiction rules return zero across 8,500 rows.**

**The absence is itself the finding.** Real banking data, entered by staff over years, normally contains some contradictions. Perfect consistency is a fingerprint of generated data.

Together with the closure rate sitting three rows off a round 20%, this is the first evidence that the dataset is synthetic. That matters later: it puts a hard ceiling on any model, and it means a high score is a property of the data rather than a sign of skill. **We say so before anyone asks.**

## Step 3 — Do leavers spend less than stayers?

Six months of card purchases per customer. Split by whether they closed the card, and take the average for each month.

`groupby("churned")` splits the 8,500 customers into the two groups. `.T` flips the table so the months run down the page instead of across.

**Month 1 is the oldest month, month 6 the most recent.** Confirmed with the manager. Everything about direction depends on that.

In [ ]:
# --- Step 3a: average purchases per month, by group ---
purchase_cols = [f"purchase_month_{i}" for i in range(1, 7)]

monthly = df.groupby("churned")[purchase_cols].mean().T
monthly.columns = ["stayed", "closed"]

monthly.round(0)

In [ ]:
# --- Step 3b: the same table as a picture ---
monthly.plot(marker="o", figsize=(7, 4))
plt.title("Average card purchases per month")
plt.xlabel("month 1 = oldest, month 6 = most recent")
plt.ylabel("average purchases")
plt.xticks(range(6), [f"month {i}" for i in range(1, 7)])
plt.grid(alpha=0.3)
plt.show()

### What we gained from Step 3

The level of spending does not separate the two groups. In the oldest month the customers who went on to close their card were spending *more* than those who stayed — 1,702 against 1,682. A model given only that month would point the wrong way.

What separates them is direction. Over six months stayers drift up about 7%; leavers fall about 27%.

Two consequences:

- Averaging the six months destroys the signal. A customer flat at 1,500 and a customer sliding 2,000 → 1,000 both average 1,500.
- The trend is not in the file. It lives across six columns and has to be built. That is what the next step does, and this chart is the justification for it.

## Step 4 — Building the trend column

The model cannot read a chart. It needs the trend as one number per customer.

For each customer, fit a straight line through their six monthly purchase figures. The **slope** of that line is how much their spending changes per month — negative for falling, positive for growing.

Six alternatives were tested for measuring the trend and all scored within 0.784 to 0.806. The slope was chosen because it uses all six months, so one odd month cannot distort it.

In [ ]:
# --- Step 4: purchase_slope — one number for the direction of spending ---
months = np.array([1, 2, 3, 4, 5, 6])

def slope_of(row):
    return np.polyfit(months, row.values, 1)[0]

df["purchase_slope"] = df[purchase_cols].apply(slope_of, axis=1)

df.groupby("churned")["purchase_slope"].mean().round(1)

### What we gained from Step 4

Six columns became one: `purchase_slope`, the change in spending per month.

Average slope for customers who stayed: **+22**. For customers who closed: **−93**. The two groups sit on opposite sides of zero — the separation that no single month could produce.

This is a group average, not a rule about individuals. Many leavers have a rising slope and many stayers a falling one. The slope moves the odds; it does not decide the outcome.

## Step 5 — Does the slope actually separate customers?

Two group averages are not proof. Sort all 8,500 customers by `purchase_slope`, cut them into six equally sized groups, and measure the closure rate in each.

A real signal produces a steady climb from one end to the other. A fake one produces a jumble.

`qcut` cuts into groups of equal *size*, not equal width — so each band holds about 1,417 customers and the percentages are comparable.

In [ ]:
# --- Step 5: closure rate across slope bands ---
df["slope_band"] = pd.qcut(
    df["purchase_slope"], 6,
    labels=["steep fall", "clear fall", "slight fall",
            "flat", "growing", "strong growth"],
)

df.groupby("slope_band", observed=True).agg(
    customers=("churned", "size"),
    closed_per_100=("churned", lambda s: round(100 * s.mean(), 1)),
)

### What we gained from Step 5

Sorting all 8,500 customers by `purchase_slope` and cutting them into six equal groups gives closure rates of:

**48.6 → 33.4 → 19.1 → 8.6 → 6.2 → 3.9** per 100.

The rate falls at every step with no reversals. That steady progression is the evidence that the column is real and not an artefact of the group averages.

The steepest-falling sixth of customers contains roughly 41% of all card closures — a targeting rule the marketing team can act on directly.

The banding is for reading and presenting only. The model receives the raw slope, because a 40% fall is genuinely worse than a 12% fall and bands would throw that away.

## Step 6 — The trend as a percentage

`purchase_slope` is what the model uses. It means nothing to a person. "Slope −93" explains nothing; "spending down 31%" explains everything.

So the same trend is built a second way, for humans: **the average of months 4–6 compared against the average of months 1–3.**

Halves, not month 6 against month 1. That distinction matters — month 6 against month 1 does not reproduce the figures below.

In [ ]:
# --- Step 6a: purchase_pct_change — the trend in plain language ---
first_half = df[purchase_cols[:3]].mean(axis=1)
last_half  = df[purchase_cols[3:]].mean(axis=1)

df["purchase_pct_change"] = (last_half - first_half) / first_half

df.groupby("churned")["purchase_pct_change"].mean().round(3)

In [ ]:
# --- Step 6b: closure rate by size of the fall ---
fall_bands = pd.cut(
    df["purchase_pct_change"],
    bins=[-np.inf, -0.30, -0.20, -0.10, -0.05, 0, np.inf],
    labels=["fell >30%", "fell 20-30%", "fell 10-20%",
            "fell 5-10%", "roughly flat", "growing"],
)

df.groupby(fall_bands, observed=True).agg(
    customers=("churned", "size"),
    closed_per_100=("churned", lambda s: round(100 * s.mean(), 1)),
)

In [ ]:
# --- Step 6c: chart — closure rate by size of the spending fall ---
rates = df.groupby(fall_bands, observed=True)["churned"].mean() * 100

ax = rates.plot(kind="bar", figsize=(7, 4), rot=20, color="indianred")
ax.set_title("Closure rate by size of the spending fall")
ax.set_ylabel("closed per 100 customers")
ax.grid(axis="y", alpha=0.3)
plt.show()

### What we gained from Step 6

Average change in card spending: stayers **+3.8%**, leavers **−18.0%**.

Closure rate by size of the fall, out of 100 customers:

- fell more than 30% (680 customers) → **70.0**
- fell 20 to 30% (784) → **42.0**
- fell 10 to 20% (1,226) → **28.1**
- fell 5 to 10% (782) → **19.1**
- roughly flat (794) → **13.4**
- growing (4,234) → **6.9**

Two things follow. The 8% of customers whose spending fell more than 30% contain about 28% of all closures — a direct targeting rule.

And the rate accelerates rather than rising evenly: 7, 13, 19, 28, 42, 70. Collapsing this into a yes/no "declining" flag would treat the 70-per-100 customer and the 19-per-100 customer identically. **This is the evidence for decision 10 — the trend stays a number, never a flag.**

## Step 7 — Do customers pay off their card before closing it?

An idea worth testing: someone who plans to close their card probably clears what they owe first. You cannot walk away from a card with money still on it.

If that is true, payments should go **up** in the months before closure, even while spending falls.

In [ ]:
# --- Step 7a: does the amount paid rise before closure? ---
payment_cols = [f"payment_month_{i}" for i in range(1, 7)]

df["payment_slope"] = df[payment_cols].apply(slope_of, axis=1)

df.groupby("churned")["payment_slope"].mean().round(1)

In [ ]:
# --- Step 7b: payments as a share of purchases, early half against late half ---
early = df[payment_cols[:3]].sum(axis=1) / df[purchase_cols[:3]].sum(axis=1)
late  = df[payment_cols[3:]].sum(axis=1) / df[purchase_cols[3:]].sum(axis=1)

df["payment_ratio"] = df[payment_cols].sum(axis=1) / df[purchase_cols].sum(axis=1)

pd.DataFrame({"first 3 months": early.groupby(df["churned"]).mean(),
              "last 3 months":  late.groupby(df["churned"]).mean()}).round(3)

In [ ]:
# --- Step 7c: the ratio still carries a signal, as a trait rather than a trend ---
df.groupby(pd.qcut(df["payment_ratio"], 5), observed=True).agg(
    customers=("churned", "size"),
    closed_per_100=("churned", lambda s: round(100 * s.mean(), 1)),
)

### What we gained from Step 7

The idea was wrong as stated. Payments do not rise before closure, and the share paid back never changes — stayers pay back about 87 of every 100 they spend in both halves of the year, leavers about 81 in both.

But the share itself turned out to matter. Closure rate across five bands, lowest share paid back first:

**40.2 → 19.2 → 15.8 → 13.5 → 11.1** per 100.

Customers who pay back the least close their card about four times as often as those who pay back the most.

So the idea pointed at the right column for the wrong reason. It is not about *when* — customers settling up before they go. It is about *who* — customers who carry debt on the card are more likely to close it. `payment_ratio` is kept for that reason.

**Failed ideas stay in this notebook on purpose.** They are the evidence that the findings which survived were actually tested.

## Step 8 — Do household commitments hold a card in place?

Not about being married. About how many people depend on the card working.

A household puts costs on the card every month by default — school fees, the shop, bills already set up. Closing it means unpicking all of that. A single customer with no dependants has far less to unpick.

`responsibility` counts the commitments: married plus has dependants, so 0, 1 or 2.

In [ ]:
# --- Step 8a: the four family situations ---
df["responsibility"] = df["married"] + df["has_dependents"]

situation = (df["married"].map({0: "single", 1: "married"})
             + df["has_dependents"].map({0: ", no children", 1: ", with children"}))

df.groupby(situation).agg(
    customers=("churned", "size"),
    closed_per_100=("churned", lambda s: round(100 * s.mean(), 1)),
)

In [ ]:
# --- Step 8b: the same thing as a single score ---
df.groupby("responsibility").agg(
    customers=("churned", "size"),
    closed_per_100=("churned", lambda s: round(100 * s.mean(), 1)),
)

### What we gained from Step 8

Out of 100 customers, how many closed their card:

- single, no children → **29**
- married, no children → **24**
- single, with children → **21**
- married with children → **16**

The claim holds. More commitments on the card, fewer closures. Children matter slightly more than marriage — 21 against 24.

As a score of 0, 1 or 2: **29 → 22.5 → 15.6**

Not just the spending trend again. Among customers falling at the same rate, the gap stays — about 52 against 43 per 100 for the steepest decliners.

**This is also where a rejected idea produced a good column.** The original hypothesis was that gender would matter for a married man with children. It does not — married with children, women 15.7 and men 15.5. But the reasoning about the mechanism was right, and `responsibility` came out of it.

**Limit:** the file already has `married` and `has_dependents` separately, and a model can combine them itself. So this adds little to the prediction. It is kept because it makes the finding sayable in one line, and gives the agent one clean thing to reason about.

## Step 9 — Three facts about the customer

Everything so far measured behaviour over six months. These three are things we already know about the person.

**Missed a loan payment before.** Someone who has struggled to pay us before is under financial pressure.

**Where the salary lands.** If it arrives in our bank, we are their main bank. If not, we are a side account.

**Holds a card elsewhere.** Direct competition for this specific product.

One trap: "never missed a payment" includes people who never borrowed at all. Two different customers stacked into one group, so they get split apart.

In [ ]:
# --- Step 9a: the three facts, one at a time ---
for column in ["missed_loan_payment_ever", "salary_lands_in_bank",
               "has_other_credit_cards"]:
    print(df.groupby(column).agg(
        customers=("churned", "size"),
        closed_per_100=("churned", lambda s: round(100 * s.mean(), 1)),
    ), "\n")

In [ ]:
# --- Step 9b: "never missed" hides two different groups ---
loan_history = np.where(
    df["had_loan_ever"] == 0, "never had a loan",
    np.where(df["missed_loan_payment_ever"] == 1,
             "had loan, missed", "had loan, never missed"))

df.groupby(loan_history).agg(
    customers=("churned", "size"),
    closed_per_100=("churned", lambda s: round(100 * s.mean(), 1)),
)

In [ ]:
# --- Step 9c: relationship_depth, 0 to 3 ---
df["relationship_depth"] = (df["salary_lands_in_bank"]
                            + (1 - df["has_other_credit_cards"])
                            + (1 - df["missed_loan_payment_ever"]))

df.groupby("relationship_depth").agg(
    customers=("churned", "size"),
    closed_per_100=("churned", lambda s: round(100 * s.mean(), 1)),
)

### What we gained from Step 9

Out of 100 customers, how many closed their card:

- missed a loan payment → **46**, never missed → **15**
- salary paid elsewhere → **29**, salary with us → **13**
- has a card elsewhere → **24**, does not → **17**

Splitting the loan column properly changes the picture. Customers who borrowed and never missed a payment are the safest group of all at **13**, safer than customers who never borrowed at **18**. A proven repayment record beats no record.

The three combined into one score from 0 to 3:

- 0 → **64.9** (208 customers)
- 1 → **35.9** (1,993)
- 2 → **17.6** (3,866)
- 3 → **6.9** (2,433)

Nearly ten times the closure rate from the weakest relationship to the strongest.

**Caution:** only 208 customers score zero. A group that small gives an unstable percentage, so treat 64.9 as approximate.

**The salary column is the most useful of the three.** Missed payments and cards elsewhere describe a situation the bank cannot change. A salary can be moved — so that one points at something to actually do.

## Step 10 — How much are they spending now?

Two customers whose spending is falling at exactly the same rate. One has dropped from 5,000 a month to 3,000. The other from 800 to 300.

The trend says they are identical. They are not. The first still puts real money through the card every month.

`recent_purchases` is the average of the three most recent months — the level, not the direction.

In [ ]:
# --- Step 10a: recent_purchases on its own ---
df["recent_purchases"] = df[purchase_cols[3:]].mean(axis=1)

print("closure rate across five groups, lowest spenders first:")
print((df.groupby(pd.qcut(df["recent_purchases"], 5), observed=True)["churned"]
         .mean() * 100).round(1).values)

In [ ]:
# --- Step 10b: does it add anything the spending trend does not already say? ---
slope_band5 = pd.qcut(df["purchase_slope"], 5,
                      labels=["1 steep fall", "2", "3", "4", "5 growth"])

level = pd.Series(np.where(df["recent_purchases"] > df["recent_purchases"].median(),
                           "spends more", "spends less"), index=df.index)

(df.pivot_table(index=slope_band5, columns=level, values="churned",
                aggfunc="mean", observed=True) * 100).round(1)

### What we gained from Step 10

On its own, out of 100 customers: **30.9 → 23.3 → 19.8 → 15.1 → 10.7**, from the lowest spenders to the highest. Three times the closure rate from one end to the other.

Held against the spending trend, the gap survives — but only where it matters:

- steepest fallers → spends less **59.4**, spends more **35.0**
- second band → **33.7** against **19.1**
- middle band → **13.3** against **13.5**, no gap
- growing customers → almost no gap

**Among customers whose spending is collapsing at the same rate, the ones already down to small amounts close at nearly twice the rate.** Among stable or growing customers it makes no difference at all.

So the column earns its place, and it earns it in a specific place — it sharpens the group that is already at risk. It is not a general-purpose signal.

## Step 11 — Is their spending steady or jumpy?

A steady spender has the card built into a routine — the same bills, the same shops, every month. An erratic spender pulls it out occasionally for one-offs.

**Habit is what keeps a card.** Someone with a routine has something to break.

`purchase_volatility` is how much their spending moves around, measured against their own average so a big spender and a small spender are comparable.

**A worry before we start:** a falling series is mechanically more variable, so this column might just be the trend in disguise. Step 11b is the check.

In [ ]:
# --- Step 11a: purchase_volatility on its own ---
df["purchase_volatility"] = df[purchase_cols].std(axis=1) / df[purchase_cols].mean(axis=1)

print("closure rate across five groups, steadiest first:")
print((df.groupby(pd.qcut(df["purchase_volatility"], 5), observed=True)["churned"]
         .mean() * 100).round(1).values)

In [ ]:
# --- Step 11b: is it just the trend in disguise? ---
jumpiness = pd.Series(np.where(df["purchase_volatility"] > df["purchase_volatility"].median(),
                               "jumpy", "steady"), index=df.index)

table = (df.pivot_table(index=slope_band5, columns=jumpiness, values="churned",
                        aggfunc="mean", observed=True) * 100)

ax = table.plot(kind="bar", figsize=(7, 4), rot=0, color=["indianred", "steelblue"])
ax.set_title("Jumpy spending matters only when spending is falling")
ax.set_ylabel("closed per 100 customers")
ax.grid(axis="y", alpha=0.3)
plt.show()

table.round(1)

### What we gained from Step 11

On its own, out of 100 customers: **12.0 → 13.1 → 15.3 → 18.2 → 41.2**, from steadiest to jumpiest. The jumpiest fifth closes at more than three times the rate of the steadiest.

**The worry was reasonable and turned out to be wrong.** Held against the spending trend, the gap not only survives — it is the largest of any column tested:

- steepest fallers → jumpy **52.0**, steady **22.5**
- second band → **38.7** against **20.9**

**And it reverses at the other end.** Among growing customers, jumpy spenders close at **3.4** and steady ones at **6.1** — the opposite direction.

That makes sense. Falling and erratic is a customer drifting out of the habit. Growing and erratic is a customer making occasional large purchases, which is a healthy card. **The same measurement means opposite things depending on which way spending is heading.**

This is the clearest example in the project of a combination that a single rule cannot express, and it is why the interaction columns in Step 15 exist.

## Step 12 — How much of their income goes through the card?

Spending on its own does not tell you much. 2,000 a month is heavy use for someone earning 6,000 and light use for someone earning 40,000.

`spend_to_salary` is average monthly spending divided by salary. It measures **engagement rather than wealth** — how woven into their life the card is.

In [ ]:
# --- Step 12a: spend_to_salary on its own ---
df["spend_to_salary"] = df[purchase_cols].mean(axis=1) / df["salary"]

print("closure rate across five groups, smallest share of income first:")
print((df.groupby(pd.qcut(df["spend_to_salary"], 5), observed=True)["churned"]
         .mean() * 100).round(1).values)

In [ ]:
# --- Step 12b: does it add anything beyond the spending trend? ---
share = pd.Series(np.where(df["spend_to_salary"] > df["spend_to_salary"].median(),
                           "high share", "low share"), index=df.index)

(df.pivot_table(index=slope_band5, columns=share, values="churned",
                aggfunc="mean", observed=True) * 100).round(1)

### What we gained from Step 12

On its own, out of 100 customers: **27.8 → 22.1 → 22.3 → 18.6 → 9.1**, from the smallest share of income through the card to the largest.

Held against the spending trend, the gap survives where it matters:

- steepest fallers → low share **55.9**, high share **38.1**
- second band → **33.2** against **21.4**
- middle band and above → no gap at all

**Among customers whose spending is collapsing at the same rate, the ones for whom the card was never a big part of their spending close far more often.** A card that carried 3% of someone's income was never load-bearing. One carrying 40% is woven into how they live.

**One caution to check later.** This measures something close to what `recent_purchases` measures — both are versions of "how much do they spend". If the model leans on one and ignores the other, that is the reason.

Also worth stating: salary on its own predicts nothing. Its correlation with closing the card is **−0.006**. Closing a card is not about affordability, it is about engagement — which is exactly why the ratio works where the raw number does not.

## Step 13 — How stretched are they?

`loan_burden` is the outstanding loan balance divided by salary. Someone owing twice their monthly salary is in a different position from someone owing twenty times it.

**One problem visible before we start.** 75% of customers have a loan balance of exactly zero. For three quarters of the book this column will be a single repeated value.

We build it anyway and test it properly. A column that fails its test is evidence the ones that passed were actually tested.

In [ ]:
# --- Step 13: loan_burden ---
df["loan_burden"] = df["outstanding_loan_balance"] / df["salary"]

print("share of customers at exactly zero:",
      round(100 * (df["loan_burden"] == 0).mean(), 1), "%")
print()

stretch = pd.Series(np.where(df["loan_burden"] > 0, "has a balance", "no balance"),
                    index=df.index)

print(df.groupby(stretch)["churned"].mean().mul(100).round(1))
print()
print((df.pivot_table(index=slope_band5, columns=stretch, values="churned",
                      aggfunc="mean", observed=True) * 100).round(1))

### What we gained from Step 13

**This column does not work, and that is the finding.**

75.4% of customers have a loan balance of exactly zero. That leaves only two usable groups: **19.1** closed per 100 for those with no balance, **23.4** for those with one.

Held against the spending trend the gap is small and inconsistent:

- steepest fallers → **52.2** with a balance, **44.9** without
- second band → **31.0** against **28.3**
- middle band → **11.7** against **13.8** — the gap reverses

**A signal that changes direction between bands is not a signal.**

And `had_loan_ever` already says most of this. Because the column is three-quarters zeros, it is close to a yes/no "do they have a loan", which the model already has.

`loan_burden` is built, tested and reported. It is a candidate for removal, and Phase 3 will show whether removing it changes anything.

## Step 14 — How often do they borrow?

`number_of_loans` is the only column in the file that nothing has been built from. It counts how many loans this customer has taken from us, from 0 to 6.

On its own the count is hard to read. Three loans means something different for a customer of two years than for a customer of twenty.

**`borrowing_rate` is loans divided by years with the bank** — how often they come to us to borrow. The "+1" in the divider is there because 6.1% of customers have zero years, and dividing by zero breaks the column.

Two things get tested, in this order. Does it separate customers on its own? And does it still say something once the spending trend is accounted for?

In [ ]:
# --- Step 14a: borrowing_rate on its own ---
df["borrowing_rate"] = df["number_of_loans"] / (df["loyalty_years"] + 1)

print("share of customers at exactly zero:",
      round(100 * (df["borrowing_rate"] == 0).mean(), 1), "%")
print()

bands = pd.qcut(df["borrowing_rate"], 5, duplicates="drop")

df.groupby(bands, observed=True).agg(
    customers=("churned", "size"),
    closed_per_100=("churned", lambda s: round(100 * s.mean(), 1)),
)

In [ ]:
# --- Step 14b: does it survive the spending trend? ---
borrows = pd.Series(np.where(df["borrowing_rate"] > df["borrowing_rate"].median(),
                             "borrows often", "borrows rarely"), index=df.index)

falling = pd.Series(np.where(df["purchase_slope"] < df["purchase_slope"].median(),
                             "spending falling", "spending steady"), index=df.index)

corner = (df.pivot_table(index=falling, columns=borrows, values="churned",
                         aggfunc="mean", observed=True) * 100)

ax = corner.plot(kind="bar", figsize=(7, 4), rot=0, color=["steelblue", "indianred"])
ax.set_title("Borrowing rate only matters when spending is falling")
ax.set_ylabel("closed per 100 customers")
ax.grid(axis="y", alpha=0.3)
plt.show()

corner.round(1)

### What we gained from Step 14

On its own, out of 100 customers, from the least frequent borrower to the most: **17.4 → 16.6 → 21.7 → 26.6**.

It reaches four bands instead of five because `number_of_loans` is a whole number from 0 to 6 and too many customers tie. That is a real weakness of the column and it is stated rather than hidden.

**The 2×2 is where it earns its place.** Out of 100 customers:

- spending steady, borrows rarely → **6.8**
- spending steady, borrows often → **5.5**
- spending falling, borrows rarely → **29.0**
- spending falling, borrows often → **37.7**

When spending is steady, the borrowing rate changes almost nothing — 6.8 against 5.5. When spending is falling it moves the rate by nearly nine points.

**The column only matters in one corner.** That is the definition of an interaction, and it means the useful version of this column is not `borrowing_rate` on its own but `borrowing_rate` crossed with the spending trend. Step 15 builds that.

**Two ideas were tested here and rejected.** The same 2×2 was run for `loyalty_years` and for `age`, and both were flat — 35.2 against 31.5, and 35.3 against 31.6. Neither interacts with the trend, so neither becomes a partner.

**A third was rejected before it was built.** `loyalty_years / (age − 18)`, the share of adult life spent with the bank, gives closure rates of 24.2, 21.0, 20.0, 16.1, 18.6 — it rises again at the end. The plain `loyalty_years` column the model already has gives 25.2, 22.0, 21.1, 17.6, 13.5, which falls at every step. **The ratio was worse than the raw column, so it was dropped.**

## Step 15 — Combinations the model cannot find on its own

Phase 1 found several effects that only appear when two things are true together:

- a missed loan payment **and** falling spending → 52 closed per 100, where neither alone explains it
- jumpy spending **and** falling spending → 52, against 22.5 for steady spenders falling just as fast
- borrowing often **and** falling spending → 37.7, against 29.0

A tree finds these by splitting twice. **A linear model cannot** — it adds columns together and can never multiply two of them. So the combinations have to be handed to it directly.

Each one is the spending trend multiplied by another column. Both are converted to **percentile ranks** first — a customer's position from 0 to 1 among all customers — so that two columns measured in completely different units can be multiplied without one swamping the other.

Eleven partners, so eleven new columns.

**The ranks have to be learned from the training customers only.** That is done properly in Phase 2 Step 6. Here we build them on everything just to see the size of the effect.

In [ ]:
# --- Step 15: the eleven interaction columns ---
PARTNERS = ["relationship_depth", "payment_ratio", "purchase_volatility",
            "missed_loan_payment_ever", "recent_purchases", "responsibility",
            "spend_to_salary", "has_other_credit_cards",
            "salary_lands_in_bank", "iscore", "borrowing_rate"]

def rank_0_to_1(s):
    return s.rank(pct=True)

slope_rank = rank_0_to_1(df["purchase_slope"])

for partner in PARTNERS:
    df["slope_x_" + partner] = slope_rank * rank_0_to_1(df[partner])

print(len(PARTNERS), "interaction columns built")
print([c for c in df.columns if c.startswith("slope_x_")])

### What we gained from Step 15

Eleven columns, each pairing the spending trend with another feature.

Measured previously with five-fold cross-validation using logistic regression, with the first ten:

- without them → AUC 0.8002, best F1 0.5434
- **with them → AUC 0.8068, best F1 0.5489**

They are worth nothing to a tree, which already finds these combinations by splitting twice. They are worth a great deal to a linear model, which cannot.

**This is why the same dataset needs different features for different models.** Feature engineering is not a fixed step done once — it depends on what the model is capable of working out for itself.

The eleventh partner, `borrowing_rate`, is new. Phase 3 measures whether it earns its place.

---

**Phase 1 is finished.** Nine columns were built and one — `loan_burden` — failed its test and is reported as a failure. Three further ideas were tested and rejected: gender within family situation, tenure as a share of adult life, and the raw `loyalty_years` and `age` as interaction partners.

---

# Phase 2 — Preprocessing

The analysis is done. This part turns the table into something a model can be trained on, and it is written for **logistic regression specifically**.

That matters. Phase 1 was planned around a tree, and trees need almost no preprocessing — they only care about the order of values, never their size. Logistic regression multiplies every value by a weight and adds them up, so the size of every number matters. A different model needs different preparation.

Six steps:

1. Choose which columns go in
2. Check that no column is cheating
3. Check the extreme values
4. Check the skew
5. Decide what to do about the 80/20 imbalance
6. Build the preparation as one reusable object
7. Build the preprocessing pipeline

## Step 1 — Choosing the columns

Not every column belongs in the model. Some are names rather than facts, some repeat what another column already says, and one is the answer itself.

Four come out for reasons already settled:

- `customer_id` — a different value for every customer, so it describes nobody
- `gender` — no signal, and banking regulation restricts using it in credit and offer decisions
- `churned` — the answer
- `slope_band`, `purchase_pct_change`, `payment_slope` — built for reading and explaining, not for the model

Two more are **suspected duplicates** and get proved rather than assumed.

In [ ]:
# --- Phase 2, Step 1: checking the suspected duplicates ---
same = (df["is_paying_old_loan"] == (df["outstanding_loan_balance"] > 0).astype(int))
print("is_paying_old_loan against outstanding_loan_balance > 0")
print("  rows where they disagree:", int((~same).sum()), "out of", len(df))
print()

print("each payment month against its purchase month:")
for i in range(1, 7):
    r = df[f"payment_month_{i}"].corr(df[f"purchase_month_{i}"])
    print(f"  month {i}: {r:.3f}")

### What we gained from Step 1

**`is_paying_old_loan`** disagrees with `outstanding_loan_balance > 0` in **zero rows out of 8,500**. It is that column rewritten as a yes/no. It comes out. The balance stays, because it also carries the amount.

**The six payment months** correlate with their purchase month between **0.978 and 0.986**. They are almost the same number twice. All six come out, and `payment_ratio` is kept in their place — it holds the only thing about payments the purchase columns do not already say.

Two removals proved rather than assumed. The rest were decided earlier and are recorded here so the reasons stay attached to the decision instead of living in someone's memory.

## Step 2 — Is any column cheating?

A column leaks when it holds information that only exists *because* the customer closed their card. The model looks brilliant in testing and fails in real life, because for a live customer that information does not exist yet.

The suspect is `salary_lands_in_bank`. If it was recorded *after* people left, it might be describing the outcome rather than predicting it.

**The method: assume the worst case, work out what the data would look like if it were true, then check.**

In [ ]:
# --- Phase 2, Step 2: is any column cheating? ---
leavers = df[df["churned"] == 1]
monthly_cols = purchase_cols + payment_cols

print("leavers:", len(leavers))
print("leavers whose salary still lands with us:",
      int(leavers["salary_lands_in_bank"].sum()),
      f"({100 * leavers['salary_lands_in_bank'].mean():.0f}%)")
print()
print("zero values anywhere in the twelve monthly columns:",
      int((df[monthly_cols] == 0).sum().sum()))
print("lowest month-6 purchase among leavers:",
      round(leavers["purchase_month_6"].min(), 2))
print()
print("leavers with more than 10 years with the bank:",
      int((leavers["loyalty_years"] > 10).sum()))

### What we gained from Step 2

**594 of 1,697 leavers still have their salary landing with us — 35%.** If the column recorded the outcome, that would be near zero. Closing a credit card does not stop a salary arriving. Not leakage.

**No zeros anywhere in the twelve monthly columns.** If the card went dead before closure, leavers would show empty final months. The lowest month-6 purchase among leavers is 55.7, not 0. These columns describe activity, not closure.

**109 leavers had been with the bank more than 10 years.** Churn is not only new customers, so tenure is not standing in for the answer either.

No leakage found. Worth stating plainly, because **a model scoring far above the published range for bank churn is usually a leakage bug rather than a triumph.** Published work on bank churn lands at AUC 0.71 to 0.85.

## Step 3 — Extreme values

An outlier is a value far away from the rest. A customer earning 3,000,000 when most earn 30,000 to 70,000.

Two questions, and they are different:

1. **Is the value wrong?** A typing mistake, a negative age. Those get fixed.
2. **Is the value unusual but real?** A genuinely wealthy customer. Those stay.

Step 2 already showed every range is plausible, so nothing here is wrong. The question is what to do with values that are real but extreme.

**And the answer depends on the model.** A tree asks "is salary above 20,000?" — a customer at 70,000 lands on the same side as one at 25,000, and the size never enters the calculation. Logistic regression multiplies the actual number, so one extreme value drags the whole coefficient.

**So this step mattered little when the plan was a tree and matters a great deal now.**

In [ ]:
# --- Phase 2, Step 3a: the standard IQR check, and where it breaks ---
def iqr_flags(column):
    q1, q3 = df[column].quantile([0.25, 0.75])
    gap = q3 - q1
    low, high = q1 - 1.5 * gap, q3 + 1.5 * gap
    return ((df[column] < low) | (df[column] > high)).sum(), round(low, 1), round(high, 1)

for c in ["salary", "outstanding_loan_balance", "iscore", "loyalty_years", "number_of_loans"]:
    n, low, high = iqr_flags(c)
    print(f"{c:26s} flagged {n:>5}  ({100*n/len(df):4.1f}%)   bounds {low} to {high}")

In [ ]:
# --- Phase 2, Step 3b: why the method fails on outstanding_loan_balance ---
c = "outstanding_loan_balance"
q1, q3 = df[c].quantile([0.25, 0.75])

print("customers with a balance of exactly zero:", round(100 * (df[c] == 0).mean(), 1), "%")
print("Q1:", round(q1), " median:", round(df[c].median()), " Q3:", round(q3))
print("so the gap between Q1 and Q3 is", round(q3 - q1), "and the upper bound is", round(q3 + 1.5*(q3-q1)))

In [ ]:
# --- Phase 2, Step 3c: are the extreme customers different? ---
for c, label in [("iscore", "lowest iscore"), ("loyalty_years", "longest tenure")]:
    q1, q3 = df[c].quantile([0.25, 0.75])
    gap = q3 - q1
    flagged = (df[c] < q1 - 1.5*gap) | (df[c] > q3 + 1.5*gap)
    if flagged.sum():
        print(f"{label:16s} {int(flagged.sum()):>4} customers -> "
              f"{100*df.loc[flagged,'churned'].mean():.0f} closed per 100, "
              f"against {100*df.loc[~flagged,'churned'].mean():.0f} for everyone else")

### What we gained from Step 3

**The standard method breaks on one column.** 75.4% of customers have a loan balance of exactly zero, so Q1, the median and Q3 are all zero. That makes the gap zero and the upper bound zero, which flags every customer with any balance at all — 2,088 people, 24.6% of the book. Having a loan is not an anomaly. **The IQR method assumes values are spread out; this column is three-quarters zeros.**

**The extreme customers are the ones that matter most.** Out of 100:

- low `iscore` (26 customers, 385 to 446) → **54 closed**, against 20 for everyone else
- long tenure (380 customers, 13.8 to 33 years) → **11 closed**, against 20

Being unusual is itself a strong signal, in both directions. **Removing these customers would remove the clearest examples of both the highest and lowest risk in the dataset.**

**So no customer is removed. Instead the extremes are pulled in.** Every column gets clipped to its 1st and 99th percentile — a value above the 99th percentile becomes the 99th percentile. The customer stays, their rank stays, only the size stops dominating.

Tested at three settings: no clipping gives F1 0.5489, clipping at 1 and 99 gives **0.5509**, clipping harder at 5 and 95 gives 0.5487. **Clipping the extreme 1% is the right amount; clipping harder removes real information.**

## Step 4 — Skew

Skew means a column has a long tail on one side. Most customers bunched at the low end and a few stretching far out to the right.

**Why it matters for this model and not for a tree.** Logistic regression assumes each column pushes the answer in a straight line. A long tail bends that line, and the model ends up fitting the tail instead of the bulk of customers.

A skew value near zero means symmetric. Above about 1 is considered strongly skewed.

Two ways to straighten a column:

- **Take the logarithm.** Simple, but it cannot handle zeros, and several of these columns are mostly zeros.
- **Yeo-Johnson.** Works out the right amount of straightening for each column separately by fitting one number per column, and it handles zeros and negatives.

The logarithm has already been tested on this data: it gave the best AUC anywhere at 0.8084, but F1 fell to 0.5484 from 0.5509. **So the logarithm is settled and not repeated.** Yeo-Johnson has not been tried, and Phase 3 tests it once.

In [ ]:
# --- Phase 2, Step 4: which columns are skewed? ---
candidates = ["salary", "outstanding_loan_balance", "loyalty_years", "iscore", "age",
              "recent_purchases", "purchase_volatility", "spend_to_salary",
              "loan_burden", "borrowing_rate", "payment_ratio"]

skew = df[candidates].skew().sort_values(ascending=False)

for name, value in skew.items():
    flag = "  <- strongly skewed" if abs(value) > 1 else ""
    print(f"{name:26s} {value:6.2f}{flag}")

### What we gained from Step 4

Skew across the columns that go into the model, most skewed first:

- `outstanding_loan_balance` → **4.08**
- `loan_burden` → **3.00**
- `borrowing_rate` → **2.94**
- `recent_purchases` → **2.43**
- `salary` → **1.94**
- `loyalty_years` → **1.71**
- `purchase_volatility` → **1.40**
- `spend_to_salary` → **0.30**
- `age` → **0.23**
- `iscore` → **−0.07**
- `payment_ratio` → **−0.86**

**Seven columns are strongly skewed and four are not.** That rules out applying one blanket transformation to everything — straightening a column that is already straight only adds noise.

Two things worth noticing. `payment_ratio` is the only column skewed the *other* way, with a long tail to the left. And **the new `borrowing_rate` from Phase 1 Step 14 is itself strongly skewed at 2.94**, which is a direct consequence of 37.8% of customers sitting at zero.

Yeo-Johnson fits its own amount per column, which is exactly the right shape of tool for a mix like this. It goes into the pipeline as a switch that can be turned off, so **Phase 3 can measure whether it earns its place rather than assuming it does.**

## Step 5 — The imbalance, and why nothing is done about it

6,803 customers stayed and 1,697 closed. Four to one.

A model tries to get as many answers right as it can. With those numbers, always guessing "stays" scores 80% — which is why accuracy is never how this model gets judged.

The usual fix is **class weighting**: tell the model each leaver counts four times as much as each stayer. The other common fix is **SMOTE**, which invents new leavers by blending real ones.

Both were tested. Neither is used, and the reason is worth stating precisely.

In [ ]:
# --- Phase 2, Step 5: what weighting actually does ---
print("stayed:", int((df.churned == 0).sum()), " closed:", int((df.churned == 1).sum()))
print("ratio:", round((df.churned == 0).sum() / (df.churned == 1).sum(), 2))
print()
print("measured previously, five-fold cross-validation:")
print("  weight 1  ->  F1 0.5236   best cut 0.26")
print("  weight 2  ->  F1 0.5251   best cut 0.39")
print("  weight 4  ->  F1 0.5238   best cut 0.56")

### What we gained from Step 5

**F1 moves by 0.0015 across the whole range of weights, while the best cut climbs from 0.26 to 0.56.**

That is the mechanism visible in numbers. Weighting inflates every predicted probability, the threshold rises to compensate, and **the same customers end up flagged.** Weighting and the threshold are two controls on one lever, and we already control the threshold directly in Phase 3.

So no weighting. The imbalance is handled by choosing where to draw the line, which is a decision the bank should make on the cost of a missed churner against the cost of a wasted offer.

**Why not SMOTE:** it invents new leavers by blending real ones, which produces nonsense in yes/no columns — a customer 0.6 married, 0.4 employed by the government. And applied before the split it leaks held-out customers into training. Weighting changes how much each real customer counts and invents nothing; SMOTE invents.

## Step 6 — The preparation, as one reusable object

Everything Phase 1 built now has to happen the same way three times: on the training customers, on the competition file, and on one customer typed into the agent in Phase 4.

If those three are prepared even slightly differently, the model gives wrong answers and **nothing crashes to warn you.** So there is one object, and it is the only way data reaches the model.

**This step also fixes a real mistake in the previous version of this notebook.**

The interaction columns use percentile ranks — a customer's position from 0 to 1 among all customers. Previously those ranks were calculated once across all 8,500 customers, before cross-validation ran. That means every fold's ranks were built partly from the customers that fold was supposed to be tested on.

It is the same mistake as scaling before the split. Mild, because it leaks the shape of the columns rather than the answer, but it inflates the score by an unknown amount.

**The fix is to make the preparation a proper scikit-learn transformer with `fit` and `transform`.** `fit` learns the ranks from training customers only; `transform` applies them. Put inside a pipeline, scikit-learn refits it separately on every fold and the held-out customers never influence it.

In [ ]:
# --- Phase 2, Step 6: the preparation transformer ---
from sklearn.base import BaseEstimator, TransformerMixin

PURCHASE = [f"purchase_month_{i}" for i in range(1, 7)]
PAYMENT  = [f"payment_month_{i}" for i in range(1, 7)]
MONTHS   = np.array([1, 2, 3, 4, 5, 6])

NEVER_USED = (["customer_id", "gender", "churned", "is_paying_old_loan",
               "slope_band", "purchase_pct_change", "payment_slope"] + PAYMENT)


class ChurnFeatures(BaseEstimator, TransformerMixin):
    '''Raw bank table in, model-ready numbers out.

    partners : which columns get crossed with the spending trend.
    '''

    def __init__(self, partners=None):
        self.partners = partners

    def _base(self, X):
        d = X.copy()
        d["purchase_slope"]      = d[PURCHASE].apply(
            lambda r: np.polyfit(MONTHS, r.values, 1)[0], axis=1)
        d["recent_purchases"]    = d[PURCHASE[3:]].mean(axis=1)
        d["purchase_volatility"] = d[PURCHASE].std(axis=1) / d[PURCHASE].mean(axis=1)
        d["payment_ratio"]       = d[PAYMENT].sum(axis=1) / d[PURCHASE].sum(axis=1)
        d["spend_to_salary"]     = d[PURCHASE].mean(axis=1) / d["salary"]
        d["loan_burden"]         = d["outstanding_loan_balance"] / d["salary"]
        d["responsibility"]      = d["married"] + d["has_dependents"]
        d["borrowing_rate"]      = d["number_of_loans"] / (d["loyalty_years"] + 1)
        d["relationship_depth"]  = (d["salary_lands_in_bank"]
                                    + (1 - d["has_other_credit_cards"])
                                    + (1 - d["missed_loan_payment_ever"]))
        return pd.get_dummies(d, columns=["employment_sector"], prefix="sector")

    def fit(self, X, y=None):
        self.partners_ = list(self.partners) if self.partners is not None else list(PARTNERS)
        d = self._base(X)
        # where every training customer sits, so a rank means the same thing later
        self.rank_ref_ = {c: np.sort(d[c].values.astype(float))
                          for c in ["purchase_slope"] + self.partners_}
        self.columns_ = [c for c in self._interactions(d).columns if c not in NEVER_USED]
        return self

    def transform(self, X):
        d = self._interactions(self._base(X))
        return d.reindex(columns=self.columns_, fill_value=0).astype(float)

    def _interactions(self, d):
        if not hasattr(self, "rank_ref_"):
            return d
        slope_rank = self._pct(d["purchase_slope"], "purchase_slope")
        for p in self.partners_:
            d["slope_x_" + p] = slope_rank * self._pct(d[p], p)
        return d

    def _pct(self, values, column):
        ref = self.rank_ref_[column]
        return np.searchsorted(ref, np.asarray(values, dtype=float), side="right") / len(ref)


# a quick look at what comes out
_probe = ChurnFeatures().fit(df)
print(len(_probe.columns_), "columns reach the model")
print(len([c for c in _probe.columns_ if c.startswith("slope_x_")]), "of them are interactions")

### What we gained from Step 6

**One object that turns the raw bank table into model-ready numbers**, used identically in training, in the competition file, and by the agent in Phase 4.

It reports the column count and the interaction count so a mismatch is visible rather than silent.

**And the rank leak is closed.** The ranks are now learned inside `fit`, which scikit-learn calls separately on every training fold. Phase 3 measures how much the old number was borrowing — expect the score to go **down** slightly. That is the correct outcome, and a score that drops when a leak is fixed is evidence the fix was real.

## Step 7 — The preprocessing pipeline

Four things happen to the numbers, in this order, and the order matters.

**1. Build the features.** The transformer from Step 6.

**2. Clip the extremes.** Pull every column in to its 1st and 99th percentile, from Step 3.

**3. Straighten the skew.** Yeo-Johnson, from Step 4. A switch, so Phase 3 can measure it.

**4. Scale.** Put every column on the same footing — subtract its average, divide by its spread. Required for logistic regression, because a column measured in tens of thousands would otherwise dominate a column measured in tenths purely because of its units.

Clipping comes before straightening because a single extreme value would otherwise decide how much straightening the whole column gets. Scaling comes last because both earlier steps change the spread.

**Every one of these is fitted inside the pipeline**, which means scikit-learn recalculates the percentiles, the straightening amounts and the averages separately for every training fold. The held-back customers never influence them.

In [ ]:
# --- Phase 2, Step 7: the preprocessing pipeline ---
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression


class ClipExtremes(BaseEstimator, TransformerMixin):
    '''Clip every column to its own 1st and 99th percentile, learned from training data only.'''

    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper

    def fit(self, X, y=None):
        Xv = np.asarray(X, dtype=float)
        self.low_  = np.quantile(Xv, self.lower, axis=0)
        self.high_ = np.quantile(Xv, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.low_, self.high_)


def build_model(partners=None, clip=0.01, straighten=False, C=0.03):
    '''The whole thing: raw table in, prediction out.'''
    steps = [
        ("features", ChurnFeatures(partners=partners)),
        ("clip",     ClipExtremes(clip, 1 - clip)),
    ]
    if straighten:
        steps.append(("straighten", PowerTransformer(method="yeo-johnson", standardize=False)))
    steps += [
        ("scale", StandardScaler()),
        ("model", LogisticRegression(C=C, max_iter=2000, random_state=42)),
    ]
    return Pipeline(steps)


print(build_model(straighten=True))

### What we gained from Step 7

**One function that builds the whole thing** — raw table in, prediction out — with four switches: which interaction partners, how hard to clip, whether to straighten, and how strong the regularisation.

Everything is fitted inside the pipeline, so nothing can leak from a held-out fold.

**Why one object rather than separate steps:** every switch is now something Phase 3 can measure instead of assume. Géron's rule is to treat data transformation choices as settings to be searched, not decisions to be guessed, and this is what makes that possible.

**Phase 2 is finished.** The preparation and the preprocessing are one pipeline, fitted per fold, with the rank leak closed.

---

# Phase 3 — The model

Phase 1 built the columns. Phase 2 built the preparation. This phase measures.

**Logistic regression only.** Earlier work surveyed about thirty model families and four linear models beat the gradient-boosted ones — logistic regression at AUC 0.8085 against XGBoost at 0.7467 on default settings. Head to head and tuned, logistic reached F1 0.5501 and XGBoost 0.5436. CatBoost was tested on the manager's question and came in at 0.7897 against 0.7900, a difference of three ten-thousandths.

Blending the two was tested across seven weightings. Their predictions correlate at **0.978** — they make the same mistakes, so averaging adds nothing.

**All of that is settled and recorded in `docs/EXPERIMENT_LOG.md`. It is not repeated here.** This phase measures only what has changed: the leak fix, the new column, and the skew step.

## Step 1 — How the model is judged

**Five-fold cross-validation on all 8,500 customers.** The customers are split into five groups. The model trains on four and predicts the fifth, five times over, so every customer gets a prediction from a model that never saw them.

**Why no separate held-out test set this time.** The earlier version of this notebook held back 20% and measured it once, which is the correct way to get a final honest number, and that number is already recorded. Choosing the model is now finished — there is nothing left to select — so the held-back fifth has no job to do, and the final model is better for being trained on all 8,500. The learning curve says going from 6,800 customers to 8,500 is worth about +0.005 AUC.

**The rule that goes with that:** because there is no fresh test set here, every number below is a cross-validated number and is reported as such. No claim is made about a test set that was already spent.

**Two measurements, and only one of them is the target.**

**AUC** answers: take one customer who closed and one who stayed, at random — how often does the model give the leaver the higher risk? 0.5 is a coin flip, 1.0 is perfect.

**F1** answers: of the customers we flag, how many actually leave, and of the customers who leave, how many did we flag? It balances the two. **F1 is what the competition scores, so F1 is the target.**

**The noise floor is 0.005, decided before anything was run.** Any difference smaller than that is the random draw, not an improvement.

In [ ]:
# --- Phase 3, Step 1: the measuring tools ---
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, confusion_matrix

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

RAW = pd.read_csv(DATA)
y = RAW["churned"].copy()
X = RAW.drop(columns=["churned"])

print(len(X), "customers,", int(y.sum()), "closed their card")


def best_f1_cut(y_true, prob):
    '''The best F1 this set of probabilities can reach, and the cut that reaches it.'''
    y_true = np.asarray(y_true)
    best_score, best_cut = 0, 0
    for c in np.arange(0.05, 0.85, 0.005):
        pred = (prob >= c).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        fn = int(((pred == 0) & (y_true == 1)).sum())
        score = 2 * tp / (2 * tp + fp + fn) if tp else 0
        if score > best_score:
            best_score, best_cut = score, c
    return best_score, best_cut


def measure(model, name):
    p = cross_val_predict(model, X, y, cv=folds, method="predict_proba")[:, 1]
    f1, cut = best_f1_cut(y, p)
    print(f"{name:38s} AUC {roc_auc_score(y, p):.4f}   F1 {f1:.4f}   at cut {cut:.3f}")
    return p, f1, cut

### What we gained from Step 1

One function that takes any version of the model and returns three numbers, so every comparison below is made the same way.

The comparison target is fixed: **AUC 0.8069 and F1 0.5509**, the previous best. Anything within 0.005 of that is unchanged.

## Step 2 — Four versions measured

Four things changed since the last run, and they are measured one at a time so we know which one did what. Changing two things at once and seeing the number move tells you nothing about which one moved it.

1. **Ten partners, ranks fitted on everything.** Reproduces the old, slightly leaky number.
2. **Ten partners, ranks fitted inside each fold.** The leak fix on its own. Expect this to go *down*.
3. **Eleven partners** — adds `borrowing_rate`.
4. **Eleven partners plus Yeo-Johnson.** The skew step.

In [ ]:
# --- Phase 3, Step 2: the four versions ---
TEN = [p for p in PARTNERS if p != "borrowing_rate"]

# 1. the old way: ranks learned from all 8,500 before cross-validation ever runs
leaky_features = ChurnFeatures(partners=TEN).fit(X)
X_leaky = leaky_features.transform(X)

leaky_model = Pipeline([("clip",  ClipExtremes(0.01, 0.99)),
                        ("scale", StandardScaler()),
                        ("model", LogisticRegression(C=0.03, max_iter=2000, random_state=42))])

p = cross_val_predict(leaky_model, X_leaky, y, cv=folds, method="predict_proba")[:, 1]
f1, cut = best_f1_cut(y, p)
print(f"{'1. ten partners, ranks leaked':38s} AUC {roc_auc_score(y, p):.4f}   F1 {f1:.4f}   at cut {cut:.3f}")

# 2, 3, 4: everything fitted inside the fold
_ = measure(build_model(partners=TEN),                          "2. ten partners, leak fixed")
_ = measure(build_model(partners=PARTNERS),                     "3. eleven partners")
_ = measure(build_model(partners=PARTNERS, straighten=True),    "4. eleven + yeo-johnson")

### What we gained from Step 2

*(Fill in from the output above — four lines, each with AUC, F1 and the cut.)*

**How to read it.** Compare each line against the previous best of AUC 0.8069 and F1 0.5509, and judge every gap against the noise floor of 0.005.

- **Line 1 against the old number** — confirms the rebuild reproduces what came before.
- **Line 2 against line 1** — the cost of the leak. A drop here is the honest correction; the old number was borrowing from the held-out customers.
- **Line 3 against line 2** — what `borrowing_rate` is worth. Under 0.005 means it did not earn its place, and that is reported as a failure rather than hidden.
- **Line 4 against line 3** — what straightening the skew is worth.

**Whichever version wins by more than the noise floor is the one that carries forward.** If nothing beats line 2, line 2 carries forward, and this step becomes a record of two ideas that were tested and did not work.

**Set `WINNER` in the next cell to whichever version won.**

In [ ]:
# --- Phase 3, Step 2b: keep the winner ---
# set these from the table above
USE_PARTNERS  = PARTNERS      # PARTNERS (eleven) or TEN
USE_STRAIGHTEN = False        # True if line 4 won by more than 0.005

final_model = build_model(partners=USE_PARTNERS, straighten=USE_STRAIGHTEN)

p_cv = cross_val_predict(final_model, X, y, cv=folds, method="predict_proba")[:, 1]
CV_F1, BEST_CUT = best_f1_cut(y, p_cv)
CV_AUC = roc_auc_score(y, p_cv)

print("chosen model")
print("  AUC:", round(CV_AUC, 4))
print("  F1 :", round(CV_F1, 4), "at cut", round(BEST_CUT, 3))

## Step 3 — Tuning the settings

Géron: *"Unless there are very few hyperparameter values to explore, prefer random search over grid search."* A grid tries every combination and wastes most of its time on settings that were never going to matter. A random search covers more ground in the same time.

And: *"treat your data transformation choices as hyperparameters."* So how hard to clip is searched alongside the model's own setting, rather than being fixed by hand.

Two settings are searched:

- **`C`** — how strongly the model is stopped from putting large weights on any one column. Small C means heavy restraint. This is also what keeps the eleven interaction columns from destabilising each other, since they are all the same slope rank multiplied by something and therefore heavily correlated by construction.
- **the clipping level** — how far in the extremes get pulled.

In [ ]:
# --- Phase 3, Step 3: random search ---
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    build_model(partners=USE_PARTNERS, straighten=USE_STRAIGHTEN),
    {"model__C":   [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10],
     "clip__lower": [0.0, 0.005, 0.01, 0.02]},
    n_iter=20, scoring="roc_auc", cv=folds,
    random_state=42, n_jobs=-1,
)

search.fit(X, y)

print("best AUC:", round(search.best_score_, 4))
print(search.best_params_)

In [ ]:
# --- Phase 3, Step 3b: measure the tuned model the same way as everything else ---
final_model = search.best_estimator_

p_cv = cross_val_predict(final_model, X, y, cv=folds, method="predict_proba")[:, 1]
CV_F1, BEST_CUT = best_f1_cut(y, p_cv)
CV_AUC = roc_auc_score(y, p_cv)

print("tuned model")
print("  AUC:", round(CV_AUC, 4))
print("  F1 :", round(CV_F1, 4), "at cut", round(BEST_CUT, 3))

### What we gained from Step 3

*(Fill in the best settings and the two numbers from the output above.)*

Tuning was the second-largest gain in the project when it was applied to XGBoost — 0.7921 to 0.8052 AUC. On logistic regression there is far less to tune, because the model has one real setting rather than six.

**A note on Adam, which the manager asked about.** Adam is a neural-network optimiser that adjusts weights by gradient descent. It is not applicable here in the way it sounds: logistic regression is solved directly by scikit-learn's own solver, and gradient boosting has no weights adjusted that way at all — it builds trees in sequence, each correcting the last. The equivalent of tuning for these models is the hyperparameter search above, which is what was run.

## Step 4 — Where to draw the line

The model outputs a probability between 0 and 1. Turning that into a decision needs a line.

**The right line depends entirely on what is being asked.**

For the competition, scored on F1, it is whichever cut maximises F1 — found by searching the cross-validated predictions across every threshold.

For the bank it is a different question, and **not ours to answer.** It depends on what a missed churner costs against what a wasted offer costs. That is open question 5 for the manager. What we can do is lay out the price list so the decision can be made on numbers.

Note the cut is around 0.25, not 0.50. **Nothing is wrong with that.** Only 20% of customers close their card, so a customer at 30% risk is well above average even though they are below half.

In [ ]:
# --- Phase 3, Step 4: the price list ---
rows = []
for cut in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    pred = (p_cv >= cut).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    rows.append({
        "cut": cut,
        "customers flagged": tp + fp,
        "churners caught": tp,
        "churners missed": fn,
        "wasted offers": fp,
        "of those flagged, % who leave": round(100 * tp / (tp + fp), 1) if tp + fp else 0,
        "of all leavers, % caught": round(100 * tp / (tp + fn), 1),
        "F1": round(2 * tp / (2 * tp + fp + fn), 4),
    })

pd.DataFrame(rows).set_index("cut")

### What we gained from Step 4

*(Fill in the chosen cut and what it costs from the table above.)*

**The table is the deliverable, not the single number.** It converts a modelling choice into a business choice: at each line, how many customers get contacted, how many leavers are caught, and how many offers are wasted on people who were never going to leave.

**The threshold is also the answer to the imbalance question from Phase 2 Step 5.** Weighting the classes and moving the threshold are two controls on the same lever — weighting inflates the probabilities and the best cut rises to compensate, flagging the same customers. Controlling the threshold directly is the simpler of the two and it is the one that can be handed to the business as a decision.

## Step 5 — Who does the model miss?

Géron: *"You should also look at the specific errors that your system makes."* This is the step most often skipped, and the only one that can find something genuinely new.

Phase 1 estimated that about 23% of customers who closed look statistically identical to customers who stayed. This step checks that against the model's actual mistakes.

The method: take the churners the model missed, and compare them column by column against the customers who stayed. If a column stands out, there is a feature still to build. If nothing stands out, the information is not in the file.

In [ ]:
# --- Phase 3, Step 5: the customers we miss ---
built = ChurnFeatures(partners=USE_PARTNERS).fit(X).transform(X)

missed = (y == 1) & (p_cv <  BEST_CUT)
caught = (y == 1) & (p_cv >= BEST_CUT)

print("caught", int(caught.sum()), "| missed", int(missed.sum()))
print()

compare = pd.DataFrame({
    "missed churners": built[missed.values].mean(),
    "caught churners": built[caught.values].mean(),
    "stayed":          built[(y == 0).values].mean(),
})
compare["missed / stayed"] = (compare["missed churners"] / compare["stayed"]).round(2)

compare.sort_values("missed / stayed").round(2)

### What we gained from Step 5

*(Fill in the caught and missed counts from the output above.)*

Previously measured: at the chosen threshold, **1,098 churners caught and 599 missed.** Comparing the missed churners to customers who stayed across all columns, almost every ratio falls between **0.86 and 1.04**. Two rows carry the finding:

- **`purchase_slope`** — missed churners **+4.73**, stayers **+22.03**, caught churners **−146.25**. The missed group is barely declining. On the strongest feature in the model they sit alongside customers who stayed.
- **`missed_loan_payment_ever`** — missed churners **0.08**, stayers **0.11**. **The churners we miss have a better payment record than the average customer who stayed.**

**No new feature is hiding in these customers.** The only ratio above 1.10 is `sector_Self-Employed` at 1.21, and that is already elevated in caught churners too and already used by the model.

**These customers left for reasons that are not in the file** — a competitor's offer, a move, a poor branch experience. None of it appears in 29 columns of card activity.

**The proper answer is not a better algorithm. It is more information** — a reason-for-closure field, competitor offer data, complaint records, branch interactions.

## Step 6 — The ceiling, demonstrated

Every check in Phase 1 Step 2 pointed at the dataset being synthetic: no missing values, no duplicates, zero contradictions across five rules, a closure rate three rows off a round 20%.

**Synthetic churn data is generated by computing a probability for each customer and then drawing the outcome at random.** That puts a hard ceiling on any model, because the coin flip itself cannot be predicted. No amount of modelling recovers a random draw.

**The test.** Take the model's predicted probabilities. Throw away the real answers. Generate fresh random answers from those probabilities, and measure AUC against them. If the model has recovered the true probability function that generated the data, the two figures match.

This is the single most important cell in the project, because it turns "we think we are near the limit" into "here is the limit."

In [ ]:
# --- Phase 3, Step 6: is the model already at the ceiling? ---
rng = np.random.default_rng(42)

scores = []
for _ in range(20):
    fake_answers = rng.binomial(1, p_cv)
    scores.append(roc_auc_score(fake_answers, p_cv))

print("AUC against regenerated answers:", round(np.mean(scores), 4),
      " (spread", round(np.std(scores), 4), "across 20 runs)")
print("AUC actually observed          :", round(CV_AUC, 4))

In [ ]:
# --- Phase 3, Step 6b: does the model know how sure it is? ---
bands = pd.qcut(p_cv, 10, labels=False)

pd.DataFrame({
    "model said": pd.Series(p_cv).groupby(bands).mean().round(3),
    "actually happened": y.groupby(bands).mean().round(3),
    "customers": y.groupby(bands).size(),
})

### What we gained from Step 6

Previously measured: **AUC against regenerated answers 0.8074**, spread 0.0069 across 20 runs, against an observed **0.8069**.

**They match. The model has recovered the process that generated the data.**

Calibration confirms it — where the model says 4 out of 100 will close, about 5 do; where it says 66 will close, 66 do. Across all ten bands the predicted and actual rates track each other.

**The ceiling is AUC about 0.807 and F1 about 0.55.** Everything remaining is the random draw.

**This closes the modelling question.** It is not an argument that the model is good enough — it is a measurement showing there is nothing left to find in this data. Further gains require information that was never recorded.

**The honest limitation to name in the report:** even a perfect model here predicts who *will* leave, not who *can be saved*. Those are different questions. A customer who leaves regardless of what you offer, and a customer who leaves *because* you contacted them, both look the same in this data. Separating them needs uplift modelling, which requires an experiment where some at-risk customers are deliberately left alone. We do not have that data. **Naming the limit is part of understanding it.**

## Step 7 — Train on everything and save

Model selection is finished, so the final model is fitted on all 8,500 customers.

It is saved **with its column list and its threshold**, so that anything scoring a customer later — the competition file, or the agent in Phase 4 — cannot get the column order wrong or invent its own cut.

In [ ]:
# --- Phase 3, Step 7: fit on all 8,500 and save ---
import joblib

final_model.fit(X, y)

bundle = {
    "model": final_model,
    "threshold": float(BEST_CUT),
    "partners": list(USE_PARTNERS),
    "columns": list(final_model.named_steps["features"].columns_),
    "cv_auc": round(float(CV_AUC), 4),
    "cv_f1": round(float(CV_F1), 4),
}

joblib.dump(bundle, "churn_model.pkl")

print("saved churn_model.pkl")
print("  columns  :", len(bundle["columns"]))
print("  threshold:", round(bundle["threshold"], 3))
print("  cv AUC   :", bundle["cv_auc"], " cv F1:", bundle["cv_f1"])

if IN_COLAB:
    files.download("churn_model.pkl")

### What we gained from Step 7

**One file that is the whole model** — the feature building, the clipping, the straightening, the scaling and the logistic regression, plus the threshold and the column list.

It takes the raw bank table as input. That is the point: the agent in Phase 4 collects raw customer details from a marketing employee and hands them straight over, with no chance of preparing them differently from the way they were prepared in training.

This file is the `predict_churn` tool that Phase 4 is built around.

## Step 8 — The competition submission

The competition file holds 1,500 customers with the same columns and no answer.

**It is scored on F1.** That was worked out from the leaderboard rather than the rules — scores cluster between 0.40 and 0.56, which is impossible for accuracy, where always predicting "stays" would already score 0.80.

**Flag a fixed number of customers rather than using a fixed probability.** Both were tested and scored the same 0.53, but the count transfers more reliably between two different sets of customers than a probability does. The best count was measured at **422 out of 1,500** across fifteen simulated draws, with a spread of 26.

Reading the test set from the leaderboard: submitting "everyone churns" scored 0.32, and since F1 = 2 × caught ÷ (flagged + actual leavers), that gives **1,500 × 0.32 ÷ (2 − 0.32) = 286 leavers**, or 19.1% of the test set against 19.96% in training.

In [ ]:
# --- Phase 3, Step 8: score the competition file ---
if IN_COLAB:
    files.upload()                       # choose "test for interns 1.csv"

test_raw = pd.read_csv("test for interns 1.csv")
print(test_raw.shape, "— expected (1500, 28)")

risk = final_model.predict_proba(test_raw)[:, 1]

print("average risk :", round(risk.mean(), 3), "(training rate is 0.200)")
print("above the cut:", int((risk >= BEST_CUT).sum()))

In [ ]:
# --- Phase 3, Step 8b: flag the top 420 and write the file ---
K = 420

order = np.argsort(risk)[::-1]           # riskiest customer first
flagged = np.zeros(len(risk), dtype=int)
flagged[order[:K]] = 1

submission = pd.DataFrame({"customer_id": test_raw["customer_id"],
                           "churned": flagged})
submission.to_csv("submission.csv", index=False)

print("flagged", int(flagged.sum()), "of", len(flagged),
      f"({flagged.mean():.1%})")
print("risk of the last customer flagged:", round(risk[order[K - 1]], 3))
submission.head()

if IN_COLAB:
    files.download("submission.csv")

### What we gained from Step 8

A submission file: one row per customer, the id and a 0 or a 1.

**How to read whatever score comes back.** The competition was simulated twelve times — hold out 1,500 customers, run the whole pipeline on the remaining 7,000, score the holdout. Cross-validated F1 averaged 0.548, test F1 averaged 0.540, and **the standard deviation of the test score was 0.024**, with the same model and the same code producing anything from 0.517 to 0.591 depending purely on which customers landed in the draw.

**So 0.53 and the leader's 0.56 are one standard deviation apart.** Both are consistent with a true score of about 0.545. A higher position on that leaderboard is mostly a kinder draw.

**And that is why there are no further submissions.** With a spread of 0.024, trying six variations would produce one scoring 0.57 by luck alone, and we would have no way to tell it from a real improvement. Géron: *"Don't tweak your model after measuring the generalization error."*

---

**Phases 1, 2 and 3 are complete.** The model is measured, its ceiling is demonstrated rather than argued, and it is saved as one file that takes raw customer details and returns a risk and a threshold.

**Phase 4 builds the agent around it.**